### Visualizing the match : radar charts

Goals : actually *see* how well a recommended song overlaps the meta-mood, instead of just trusting the similarity score.

This is the last item on my roadmap. The recommender gives me a list of songs with a cosine score, but a number doesn't really tell me *why* a song fits. A radar chart over the 8 audio features lets me eyeball it : I draw the meta-mood as one shape and a song as another, and the more they overlap, the better the match.

> I'll just reuse the engine from `src/engine.py` to get a blend and its recommendations, then plot with Plotly so the charts are interactive.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import plotly.graph_objects as go
from src.engine import MetaTune, features

mt = MetaTune()
blend = {"Stardew Valley": 0.6, "Detroit: Become Human": 0.4}
recs = mt.recommend(blend, top_n=10)
recs[["track_name", "artists", "track_genre", "sim"]]

,track_name,artists,track_genre,sim
0,Life Time Warranty,Cyberbully Mom Club,garage,0.998141
3,Long Walks,Joi Casette,study,0.996415
1,EVERYTHING,The Black Skirts,indie-pop,0.997473
6,Wishful Thinking,Earl Klugh,guitar,0.994757
2,The Girl Who Fell from the Sky,Otaku;LoFi Waiter,anime,0.996544
5,Dueling Banjos,Earl Scruggs,bluegrass,0.994814
7,Ecstasy (Instrumental Edit),Crooked Still,bluegrass,0.993539
4,Taking a Gap Year,Matt Large,study,0.994909
8,Shadows In Silence,Enigma,new-age,0.993449
11,M.,Anıl Emre Daldal,turkish,0.992438


### Drawing the mood against a song

> Both the mood and every song live in the same 0–1 feature space, so I can put them on the same radar. The closer a song's shape hugs the mood's shape, the closer it sits to the vibe I asked for.

In [2]:
mood = mt.mood(blend)

def track_vector(track_id):
    # a song's 8 features, in the same scaled space as the mood
    row = mt.songs[mt.songs.track_id == track_id][features]
    return mt.scaler.transform(row)[0]

def radar(track_id):
    song = track_vector(track_id)
    info = recs[recs.track_id == track_id].iloc[0]
    axes = features + [features[0]]   # repeat first point so the polygon closes
    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(r=list(mood) + [mood[0]], theta=axes, fill="toself", name="meta-mood"))
    fig.add_trace(go.Scatterpolar(r=list(song) + [song[0]], theta=axes, fill="toself", name=info["track_name"]))
    fig.update_layout(
        polar=dict(radialaxis=dict(range=[0, 1])),
        title=f"{info['artists']} - {info['track_name']}  (sim {info['sim']:.3f})",
    )
    return fig

radar(recs.iloc[0]["track_id"])

> The top pick wraps the mood almost perfectly, which is exactly what the high similarity was telling me, except now I can actually see it feature by feature.

### Good match vs bad match

> To be sure the radar is really showing something, let me put the **best** match next to a **bad** one : a popular song that sits as far from the mood as possible.

In [3]:
from sklearn.metrics.pairwise import cosine_similarity

# find a popular song that is far from the mood, for contrast
popular = mt.songs[mt.songs.popularity > 70].copy()
popular["sim"] = cosine_similarity(mt.scaler.transform(popular[features]), mood.reshape(1, -1)).ravel()
worst = popular.sort_values("sim").iloc[0]

axes = features + [features[0]]
good = track_vector(recs.iloc[0]["track_id"])
bad = mt.scaler.transform(worst[features].values.reshape(1, -1))[0]

fig = go.Figure()
fig.add_trace(go.Scatterpolar(r=list(mood) + [mood[0]], theta=axes, fill="toself", name="meta-mood"))
fig.add_trace(go.Scatterpolar(r=list(good) + [good[0]], theta=axes, fill="toself", name="good match"))
fig.add_trace(go.Scatterpolar(r=list(bad) + [bad[0]], theta=axes, fill="toself", name=f"bad match: {worst['track_name']}"))
fig.update_layout(polar=dict(radialaxis=dict(range=[0, 1])), title="Good vs bad match against the mood")
fig

/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


> And there's the proof : the good match sits right on top of the mood, the bad one blows out on the features the mood is low on (energy, danceability...). The radar makes it obvious which features pull a song toward the vibe and which push it away.

### Wrap up

> That closes the roadmap : I can now blend games into a mood, recommend real songs for it, export them to a playlist, and finally *see* why each song fits. From a pile of OSTs to an actual, explainable music recommender.